In [ ]:
"""
Dimensionality of the stay-period deformation in TIMBRE.

Question: when TIMBRE is given extra hidden nodes during stay (immobility),
what do they learn beyond the M3 (single-node-per-arm, in-phase) solution?

Result chain (all on stay, which_phase=2):
  1. GATE          n1 reliably relearns M3 (sim~0.98); n2 does NOT converge
                   across trainings (sim~0.64) -> n2 is not a single object.
  2. PRINCIPAL     The {n1,n2} plane has one locked axis (M3, cos~0.998 in all
     ANGLES        arms) and one wandering axis -> "M3 stable, extra wanders."
  3. DIMENSION     Pooling the wandering directions across trainings shows the
                   extra space is multi-dimensional (~3D resolved at M6), and
                   its size grows with node count -> M6 under-resolves it.
  4. INFORMATIVE   eff_dim tracks test accuracy up to ~48 nodes (~21 dims) then
     DIM           diverges (dim climbs, accuracy flat, train/test gap widens)
                   -> the deformation is informative up to ~21 dims; beyond
                      that, capacity fits noise.

Implication: TIMBRE captures this ~21-D structure by brute force, one node at
a time. A low-parameter model of how the subspace deforms could capture the
same information with far less capacity. Whether such a model exists depends on
whether the 21 dims share low-rank temporal dynamics -- the next experiment.

All rotation-invariant similarities use |<a,b>|/(||a|| ||b||): the complex
correlation, invariant to the global-phase freedom of TIMBRE's |W.x| layer.
"""

import numpy as np
from itertools import combinations
from scipy.optimize import linear_sum_assignment
from primitives import sim, align, orthobasis, principal_cos, participation_ratio, extra_subspace, partition_ok
from analysis import test_accuracy, capstone_figure


# ===========================================================================
# training -> one run's mixing matrices and arm->node maps
# ===========================================================================
def train_run(n_nodes, *, wLFPs, LFPs, lapID, test_inds, train_inds, helpers, TIMBRE):
    """
    One fresh training of M3 and M(n_nodes). Returns the mixing matrices
    (node x electrode) and arm->node assignments needed by every analysis.
    """
    def mixing(model, k):
        raw = helpers.layer_output(wLFPs, model, 0)          # (T, 2k) real
        acts = raw[:, :k] + 1j * raw[:, k:]                  # (T, k) complex
        return np.linalg.pinv(acts[train_inds]) @ LFPs[train_inds]   # (k, 192)

    def arm_map(model, top_k):
        p = helpers.layer_output(wLFPs[test_inds], model, 2)         # (n_test, k)
        labels = lapID[test_inds, 1].astype(int)
        means = np.array([p[labels == a].mean(0) for a in range(3)])
        return [list(np.argsort(means[a])[-top_k:][::-1]) for a in range(3)]

    m3, _, _ = TIMBRE(wLFPs, lapID[:, 1], test_inds, train_inds, hidden_nodes=3)
    mk, _, _ = TIMBRE(wLFPs, lapID[:, 1], test_inds, train_inds, hidden_nodes=n_nodes)
    return dict(
        W3=mixing(m3, 3), Wk=mixing(mk, n_nodes),
        arms3=arm_map(m3, 1), armsk=arm_map(mk, n_nodes // 3),
        model_k=mk,
    )


# ===========================================================================
# 1. GATE: is n1 = M3? is n2 a stable object?
# ===========================================================================
def gate(runs, arm):
    """runs: list of train_run dicts at n_nodes=6. Returns the key similarities."""
    m3 = [r['W3'][r['arms3'][arm][0]] for r in runs]
    n1 = [r['Wk'][r['armsk'][arm][0]] for r in runs]
    n2 = [r['Wk'][r['armsk'][arm][1]] for r in runs]
    cross = lambda v: np.mean([sim(a, b) for a, b in combinations(v, 2)])
    return dict(
        n1_self=cross(n1), n2_self=cross(n2),               # n1 stable? n2 stable?
        n1_to_m3=np.mean([sim(n1[i], m3[i]) for i in range(len(runs))]),
        n2_to_m3=np.mean([sim(n2[i], m3[i]) for i in range(len(runs))]),
    )


# ===========================================================================
# 2. PRINCIPAL ANGLES: is the {n1,n2} plane stable even though n2 isn't?
# ===========================================================================
def plane_stability(runs, arm):
    """Mean cosine of principal angles between {n1,n2} planes across run-pairs.
    Returns (mean over both angles, per-angle mean) -- expect ~[0.998, wandering]."""
    planes = [[r['Wk'][r['armsk'][arm][0]], r['Wk'][r['armsk'][arm][1]]] for r in runs]
    per_pair = [principal_cos(orthobasis(A), orthobasis(B))
                for A, B in combinations(planes, 2)]
    per_pair = np.array(per_pair)                            # (n_pairs, 2)
    return per_pair.mean(), per_pair.mean(0)


# ===========================================================================
# 3 & 4. DIMENSIONALITY of the extra (beyond-M3) space
# ===========================================================================
def effective_dim(runs, arm):
    """Participation ratio of all runs' extra-directions pooled together.
    Phase-invariant; >1 means the extra space is multi-dimensional."""
    cols = []
    for r in runs:
        vecs = [r['Wk'][n] for n in r['armsk'][arm]]
        Qx = extra_subspace(vecs, r['W3'][r['arms3'][arm][0]])
        cols.extend(Qx[:, c] for c in range(Qx.shape[1]))
    return participation_ratio(np.linalg.svd(np.column_stack(cols), compute_uv=False))


# ===========================================================================
# DRIVER
# ===========================================================================
def run_all(*, wLFPs, LFPs, lapID, test_inds, train_inds, helpers, TIMBRE,
            n_runs=5, sweep_counts=(6, 12, 24, 48, 96, 120)):
    """End-to-end. Returns gate/plane results per arm and eff_dim+accuracy sweep."""
    tr = lambda nc: [train_run(nc, wLFPs=wLFPs, LFPs=LFPs, lapID=lapID,
                               test_inds=test_inds, train_inds=train_inds,
                               helpers=helpers, TIMBRE=TIMBRE) for _ in range(n_runs)]

    runs6 = tr(6)
    gate_by_arm  = {a: gate(runs6, a) for a in range(3)}
    plane_by_arm = {a: plane_stability(runs6, a) for a in range(3)}

    eff_dims, test_accs, train_accs = {6: effective_dim(runs6, 0)}, {}, {}
    runs_by_count = {6: runs6}
    for nc in sweep_counts:
        runs = runs6 if nc == 6 else tr(nc)
        runs_by_count[nc] = runs
        if not partition_ok(runs[0]['armsk'], nc):
            print(f"WARNING n_nodes={nc}: arm overlap -> eff_dim may be inflated")
        eff_dims[nc] = effective_dim(runs, 0)
        test_accs[nc] = np.mean([test_accuracy(r['model_k'], wLFPs=wLFPs, lapID=lapID,
                                               test_inds=test_inds, helpers=helpers)
                                 for r in runs])
    return dict(gate=gate_by_arm, plane=plane_by_arm,
                eff_dims=eff_dims, test_accs=test_accs, runs_by_count=runs_by_count)


# ---------------------------------------------------------------------------
# USAGE
# ---------------------------------------------------------------------------
# import helpers
# from TIMBRE import TIMBRE
# # wLFPs, LFPs, lapID, test_inds, train_inds already built (which_phase=2 = stay)
#
# res = run_all(wLFPs=wLFPs, LFPs=LFPs, lapID=data['lapID'],
#               test_inds=test_inds, train_inds=train_inds,
#               helpers=helpers, TIMBRE=TIMBRE)
#
# print(res['gate'][0])     # {'n1_self':~0.98, 'n2_self':~0.64, 'n1_to_m3':~0.98, ...}
# print(res['plane'][0])    # (mean_cos, [~0.998, wandering])
#
# fig = capstone_figure(res['eff_dims'], res['test_accs'])
# fig.savefig("deformation_dimensionality.png", dpi=150)

In [ ]:
%load_ext autoreload
%autoreload 2